In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, upper

#1. Read bronze table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

#2. Silver Transformation

##2.1. Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
       

##2.2. Normalization

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status"
        , F.when(upper(F.col("cst_marital_status")) == "M", "Married")
        .when(upper(F.col("cst_marital_status")) == "S", "Single")
        .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr"
        , F.when(upper(F.col("cst_gndr")) == "M", "Male")
        .when(upper(F.col("cst_gndr")) == "F", "Female")
        .otherwise("n/a")
    )
)

##2.3. Remove records with missing customer ID

In [0]:
df = df.filter(col("cst_id").isNotNull())

# Renaming Columns

In [0]:
Rename_Map = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

for old_name, new_name in Rename_Map.items():
    df = df.withColumnRenamed(old_name, new_name)


#Sanity checks of dataframe

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

#Sanity checks of silver Table

In [0]:
%sql
select * from workspace.silver.crm_customers limit 10